###################
# SLEIPNIR - Plots
###################
# Sequential Light Echo Identification Protocol Nearly In Realtime
# Primary plotting script
###################

In [ ]:
#############################################################################################################
# User Inputs
#############################################################################################################

Sample_Name = 'Example_Data'
Run_Date = '2026-10-29'
# Must come from the same run (Sample_Name + Run_Date) as the processed spectra read from Outputs/{Sample_Name}_{Run_Date}
SLEIPNIR_Results_File_Path = 'Outputs/Example_Data_2026-10-29/Result_Files/SLEIPNIR_Results_File_for_Example_Data.ascii'
Desired_Object_Flags = [0,1,2,5,7] # All objects

###################
# Behaviour Toggles
###################

ignore_config_check = False
Line_Report = False
ignore_score_check = True

# NB: Classification_Plot, Shift_Value and zip_up are not currently used anywhere in this notebook
Classification_Plot = True
Shift_Value = 750

zip_up = False

In [ ]:
#############################################################################################################
# Imports and initial set up
#############################################################################################################

import Modules.Core.Core_Functions as Core_Functions
import Modules.Core.Config_Readers as Config_Readers
import Modules.Spectra.Spectra_Functions as Spectra_Functions
import Modules.Spectra.Line_Lists as Line_Lists
import Modules.Plotting.Plotting_Functions as Plotting_Functions

import os
import sys
import time

import numpy as np
import scipy.constants as constants
import pandas as pd

from tqdm.notebook import tqdm
from astropy import units as u

np.seterr(invalid='raise')
startT = time.time()

c = constants.value('speed of light in vacuum') / 1000  # Speed of light in kms^-1

# Marks a line as 'not usable for this spectrum' using the sentinel values the rest of the pipeline expects:
#   -999  = line (or its continuum windows) not covered by the spectrum
# Used both when the line lies outside the spectrum and when the spectrum ends inside the continuum windows.
# NB: the dictionary entries persist between objects, so every slot the plotting / results code reads is reset here.
def Mark_Line_Uncovered(line_entry):
	for slot in (2, 3, 4, 5, 6, 7, 9, 10, 11, 21, 22, 23, 24, 25, 26):
		line_entry[slot] = np.nan
	for slot in (8, 16, 17, 20):
		line_entry[slot] = -999


In [ ]:
#######################
# Configuration
#######################
# The vast majority of the code's configuration is handled by four separate configuration files:
# A cuts configuration file: Details the requirements for candidate flagging and which spectra are selected in the first place
# A processing configuration file: Deals with how the spectra are smoothed etc.
# A scoring configuration file: Details the requirements for each individual line to score
# A plots configuration file: Determines which plots are generated - Note the more plots are active the slower the script is to run, only select the ones you need
# Each file has internal documentation explaining each parameter
# These are pathed to via the following variables

Cuts_Configuration_File = "Configuration_Files/Cuts/Initial_Cuts.config"
Processing_Configuration_File = "Configuration_Files/Processing/Spectral_Processing.config"
Scoring_Configuration_File = "Configuration_Files/Scoring/ECLE_Scoring.config"
Plots_Configuration_File = "Configuration_Files/Plots/Plots_Config.config"

debug = False # If True, displays several blocks of information to assist with debugging
header_print = False # If True displays the header block info of the DESI spec files
consistency_check = False # If True prints a subsection of the flux arrays of each spectrum at every processing step
plot_spec_check = False # If True will display the first five spectra processed per file as a quick visual check of the processing

verbose = True # If True, some functions / analysis steps will provide intermediate outputs

In [ ]:
#######################
# Configuration Processing
#
# Handles the configuration files
# All of these functions will terminate the script if a configuration error is detected
#######################

# Cuts Configuration
Configured_Cuts = Config_Readers.cuts_configuration_reader(filepath=Cuts_Configuration_File)

Low_z_Cut = Configured_Cuts[0]
High_z_Cut = Configured_Cuts[1]

# Processing Configuration
Configured_Processing_Steps = Config_Readers.processing_configuration_reader(filepath=Processing_Configuration_File)

Spectres_Rebin = Configured_Processing_Steps[0]
Rebin_Res = Configured_Processing_Steps[1]

FFT_Denoise = Configured_Processing_Steps[2]
FFT_Threshold = Configured_Processing_Steps[3]

Gaussian_Smooth = Configured_Processing_Steps[4]
Gaussian_Sigma = Configured_Processing_Steps[5]

Median_Filter = Configured_Processing_Steps[6]
Median_Filter_Kernel = Configured_Processing_Steps[7]

Sav_Gol = Configured_Processing_Steps[8]
Sav_Gol_Window = Configured_Processing_Steps[9]
Sav_Gol_Poly = Configured_Processing_Steps[10]

Problematic_Skyline_Removal = Configured_Processing_Steps[11]
Harsh_Skyline_Removal = Configured_Processing_Steps[12]
Strong_Sky_Post_Rebin_Harsh_Removal = Configured_Processing_Steps[13]
Telluric_Band_Masking = Configured_Processing_Steps[14]

# Scoring Configuration

Configured_Scoring = Config_Readers.scoring_configuration_reader(filepath=Scoring_Configuration_File)

Candidate_Threshold = Configured_Scoring[0]
Candidate_Score_Max = Configured_Scoring[1]

Lower_Wave = Configured_Scoring[2]
Upper_Wave = Configured_Scoring[3]

Lower_Shift = Configured_Scoring[4]
Upper_Shift = Configured_Scoring[5]

Line_Region = Configured_Scoring[6]
Feature_Region = Configured_Scoring[7]

LineDetection_pEQW_Threshold = Configured_Scoring[8]
LineDetection_Peak_Threshold = Configured_Scoring[9]
Line_Peak_Region_Min_Threshold = Configured_Scoring[10]

Relative_SNR_Threshold = Configured_Scoring[11]
Min_Relative_SNR_Threshold = Configured_Scoring[12]
Min_Espec_SNR_Threshold = Configured_Scoring[13]
Strong_SNR_Threshold = Configured_Scoring[14]
Weak_FeVII_SNR_Threshold = Configured_Scoring[15]

# Plots Configuration

Configured_Plots = Config_Readers.plotting_configuration_reader(filepath=Plots_Configuration_File)

Co_Ord_Summary_Plot = Configured_Plots

# Wavelength or Shift
Mode = 'Shift' # Only 'Shift' is currently valid

startT = time.time()

print('Initial configuration complete')

####################
# Pathing + Config Settings Flag Generation
####################

MainSavePath = os.path.join(f"Outputs/{Sample_Name}_{Run_Date}")
Processed_Spectra_SavePath = os.path.join(MainSavePath, "Result_Files/Processed_Spectra")
Classification_Plots_SavePath = os.path.join(MainSavePath, "Result_Files/Classification_Plots")

Core_Functions.directory_creation(MainSavePath)
Core_Functions.directory_creation(Processed_Spectra_SavePath)
Core_Functions.directory_creation(Classification_Plots_SavePath)

# This config value encodes the specific set up used for each run in the results file to make this traceable.
# This is compared to the setting stored in the result files - if this does not match (and the check is not disabled) an error is raised
# NB: it must be built identically to the one in SLEIPNIR_Analysis, and does not cover the skyline / telluric switches or the scoring thresholds
# (the score comparison further down is what catches changes to those)

Settings_Config = f'{int(Spectres_Rebin)}_{Rebin_Res:g}_{int(FFT_Denoise)}_{int(FFT_Threshold)}_{int(Gaussian_Smooth)}_{Gaussian_Sigma}_{int(Median_Filter)}_{Median_Filter_Kernel}_{int(Sav_Gol)}_{Sav_Gol_Window}_{Sav_Gol_Poly}_{Low_z_Cut}_{High_z_Cut}'

print('Additional configuration and Pathing Config Complete')

In [ ]:
########################
# Line 'list' Dictionaries
########################

# The processed spectra are always saved on the air scale (see SLEIPNIR_Analysis), so only the air lists are used here
Spectral_Lines_For_Analysis = Line_Lists.lines_for_analysis_air() # All the lines here are analysed
Spectral_Lines_For_Scoring = Line_Lists.lines_for_scoring_air() # All the lines here factor into the ECLE scoring
Spec_Lines = Line_Lists.primary_lines_air() # Line keys
Spec_Names = Line_Lists.primary_line_names_air() # Fully formatted line names and wavelengths
Spec_Labels = Line_Lists.primary_line_labels_trimmed_air()
Spectral_Lines_For_Analysis_Names = list(Spectral_Lines_For_Analysis.keys()) # Display ready line labels

Data_Input = pd.read_csv(SLEIPNIR_Results_File_Path, comment='#', sep='\t')

All_Object_Data = Data_Input[Data_Input['Candidate_Flag'].isin(Desired_Object_Flags)]
All_Object_Data = All_Object_Data.reset_index(drop=True)

Number_of_Objects = len(All_Object_Data)


Read in, processing and plotting of spectral files

The processed spectra saved by `SLEIPNIR_Analysis` are read back in (they have already been extinction and redshift corrected, rebinned and smoothed, so nothing further is applied here).

The line measurements and scoring are then deliberately repeated so the plots exactly match the results file. The recalculated candidate score is compared to the recorded one and the run stops at the first mismatch (unless `ignore_score_check` is set) - this usually means the configuration or the analysis code changed after the analysis was run. A change to the measurement code in `SLEIPNIR_Analysis` has to be mirrored here.


In [ ]:
print(f'Number of Objects: {Number_of_Objects}')

for Spectrum_Number in tqdm(range(Number_of_Objects), desc="Number of spectra processed"):

	Candidate_Score = 0
	Strong_FeVII_Flag = 0
	Strong_FeX_Flag = 0
	Strong_FeXI_Flag = 0
	Strong_FeXIV_Flag = 0
	Strong_FeVII_3759_Flag = 0
	Strong_FeVII_5159_Flag = 0
	Strong_FeVII_5721_Flag = 0

	Detected_FeVII_Flag = 0
	Detected_FeX_Flag = 0
	Detected_FeXI_Flag = 0
	Detected_FeXIV_Flag = 0
	Detected_FeVII_3759_Flag = 0
	Detected_FeVII_5159_Flag = 0
	Detected_FeVII_5721_Flag = 0

	Secondary_FeVII_Line_Flag = 0

	Object_Data = All_Object_Data.iloc[Spectrum_Number]

	if not ignore_config_check:

		if Object_Data['Settings_Config'] != Settings_Config:
			print(f"Config mismatch: Stored {Object_Data['Settings_Config']}\t Current {Settings_Config}\n")
			sys.exit(1)

	Candidate_Score = 0
	Recorded_Database_Score = Object_Data['Score']

	# Per_Object_Data
	ID = Object_Data['ID']
	SpectrumID = Object_Data['Spectrum_Number']
	Redshift = Object_Data['Redshift']
	Observation_Date = Object_Data['MJD']

	Spectrum_Path = f"{Processed_Spectra_SavePath}/{ID}_{int(SpectrumID)}_Processed_Spectrum.ascii"
	Spectrum = pd.read_csv(Spectrum_Path, comment='#', sep='\t')

	Wave = list(Spectrum['Wavelength']) * u.AA
	Flux = list(Spectrum['Flux']) * u.Unit('erg cm-2 s-1 AA-1')
	FluxErr = list(Spectrum['FluxErr']) * u.Unit('erg cm-2 s-1 AA-1')

	Skylines_Retrieval = Line_Lists.sky_lines_air(z=Redshift)
	problematic_sky_list, problematic_sky_list_z_corrected = Line_Lists.problematic_skylines_air(z=Redshift, mode='DESI')

	Skylines = Skylines_Retrieval[1]

	Flux_Values = np.asarray(getattr(Flux, 'value', Flux), dtype=float)
	Err_Values = np.asarray(getattr(FluxErr, 'value', FluxErr), dtype=float)

	Per_Pixel_SNR = np.full_like(Flux_Values, np.nan)
	valid = np.isfinite(Flux_Values) & np.isfinite(Err_Values) & (Err_Values > 0)
	Per_Pixel_SNR[valid] = Flux_Values[valid] / Err_Values[valid]

	Mean_SNR = np.nanmean(Per_Pixel_SNR)

	# Line report generation if desired
	# NB: only the header row is currently written - the per-line rows are never added, so the report is otherwise empty
	if Line_Report:
		line_scoring_report = open(
			f'{Processed_Spectra_SavePath}/Line_Scoring_Report_For_{ID}.csv','w'
		)

		line_scoring_report.write(
			f'Line\tpEQW\tPeak Mean\tPeak Max\tPeak Max Location\tRegion Mean\tRegion Max\tRegion Max Location\t'
			f'Condition\n'
		)
		line_scoring_report.close()
	#################################
	# Line pEQW measurements using a linear continuum fit (the flux is divided by the continuum, not subtracted)
	# These calculations deliberately repeat the scoring processes of the analysis script to ensure the plots being displayed fully match
	# the version of the spectrum (and linked parameters) in the results files
	#################################

	for ii, item in enumerate(Spectral_Lines_For_Analysis):

		Line_Location = Spectral_Lines_For_Analysis[item][0]

		# Velocity shift from line
		Shift = (((np.array(Wave) * c) / Line_Location) - c) * (u.kilometer / u.second)

		# Cut down the region for analysis (Lower_Shift to Upper_Shift around the line, from the scoring config)
		# The line is treated as not covered if the spectrum ends more than 50 Angstroms short of it - this MUST match SLEIPNIR_Analysis

		if Wave[-1].value < Line_Location - 50 or Wave[1].value > Line_Location + 50:

			if verbose:
				print(
					f"Analysis region \n{Spectral_Lines_For_Analysis_Names[ii]} is not fully covered in the rest frame."
				)

			Mark_Line_Uncovered(Spectral_Lines_For_Analysis[item])
			continue # Uncovered lines never score - without this the scoring below used the previous line's arrays

		else:

			Shift_Region, Wave_Region, Flux_Region, Error_Region, Per_Pixel_SNR_Region = Spectra_Functions.region_cutter(
				shift=Shift,
				wave=Wave,
				flux=Flux,
				error=FluxErr,
				per_pixel_snr=Per_Pixel_SNR,
				low_cut=Lower_Shift,
				high_cut=Upper_Shift,
				mode=Mode
			)

			# Generate and remove a continuum fit from the flux data - currently linear other functions seem worse
			# The scaled flux output - Continuum[1] - is scaled to a continuum value of 1
			# The returned continuum values - Continuum[0] - is relative to the original unscaled spectrum
			Continuum = Spectra_Functions.continua_maker(
				xaxis=Shift_Region,
				flux=Flux_Region,
				line_name=list(Spectral_Lines_For_Analysis)[ii],
				line_loc=Line_Location,
				object_name=ID,
				mode=Mode
			)

			# Partial coverage: the spectrum ends inside a continuum window, so no continuum can be fitted. Treat the line as not
			# covered (as the scoring threshold modifier expects) rather than letting eqw_measurement fail on the empty continuum
			if len(Continuum[0]) == 0:
				Mark_Line_Uncovered(Spectral_Lines_For_Analysis[item])
				continue

			# Measure the resulting pEQW of the feature

			pEQW_Spec, Line_Flux, Relative_max_snr, Relative_min_snr, Espec_max_snr, Espec_min_snr, Continuum_STDEV = Spectra_Functions.eqw_measurement(
				flux=Flux_Region,
				true_continuum=Continuum[0],
				scaled_flux=Continuum[1],
				continuum_regions = Continuum[2],
				wave=Wave_Region,
				shift=Shift_Region,
				error=Error_Region,
				per_pixel_snr=Per_Pixel_SNR_Region,
				line_loc=Line_Location,
				peak_region_range=Line_Region,
				mode=Mode
			)

			# Store the results in the main dictionary

			Spectral_Lines_For_Analysis[item][2] = Shift
			Spectral_Lines_For_Analysis[item][3] = Shift_Region
			Spectral_Lines_For_Analysis[item][4] = Wave_Region
			Spectral_Lines_For_Analysis[item][5] = Flux_Region
			Spectral_Lines_For_Analysis[item][6] = Continuum[0]
			Spectral_Lines_For_Analysis[item][7] = Continuum[1]

			Spectral_Lines_For_Analysis[item][21] = Error_Region
			Spectral_Lines_For_Analysis[item][22] = Per_Pixel_SNR_Region

			if not np.isfinite(pEQW_Spec):
				Spectral_Lines_For_Analysis[item][8] = -9999
			else:
				Spectral_Lines_For_Analysis[item][8] = pEQW_Spec

			if not np.isfinite(Line_Flux):
				Spectral_Lines_For_Analysis[item][16] = -9999
			else:
				Spectral_Lines_For_Analysis[item][16] = Line_Flux

			if not np.isfinite(Relative_max_snr):
				Spectral_Lines_For_Analysis[item][17] = -9999
			else:
				Spectral_Lines_For_Analysis[item][17] = Relative_max_snr

			if not np.isfinite(Relative_min_snr):
				Spectral_Lines_For_Analysis[item][23] = -9999
			else:
				Spectral_Lines_For_Analysis[item][23] = Relative_min_snr

			if not np.isfinite(Espec_max_snr):
				Spectral_Lines_For_Analysis[item][24] = -9999
			else:
				Spectral_Lines_For_Analysis[item][24] = Espec_max_snr

			if not np.isfinite(Espec_min_snr):
				Spectral_Lines_For_Analysis[item][25] = -9999
			else:
				Spectral_Lines_For_Analysis[item][25] = Espec_min_snr

			if not np.isfinite(Continuum_STDEV):
				Spectral_Lines_For_Analysis[item][26] = -9999
			else:
				Spectral_Lines_For_Analysis[item][26] = Continuum_STDEV

		if Spectral_Lines_For_Analysis_Names[ii] in Spectral_Lines_For_Scoring:

			LineScore = Spectra_Functions.lin_con_candidate_line_identifier(
				line_name=Spectral_Lines_For_Analysis_Names[ii],
				line_peqw=Spectral_Lines_For_Analysis[item][8],
				peqw_threshold=LineDetection_pEQW_Threshold,
				continuum_removed_flux=Continuum[1],
				shift_points=Shift_Region,
				peak_threshold=LineDetection_Peak_Threshold,
				peak_region=Line_Region,
				feature_region=Feature_Region,
				peak_region_minima_threshold=Line_Peak_Region_Min_Threshold,
				relative_max_snr=Relative_max_snr,
				relative_min_snr=Relative_min_snr,
				espec_min_snr=Espec_min_snr,
				relative_snr_threshold=Relative_SNR_Threshold,
				min_relative_snr_threshold=Min_Relative_SNR_Threshold,
				min_espec_snr_threshold=Min_Espec_SNR_Threshold,
				strong_snr_threshold=Strong_SNR_Threshold,
				weak_FeVII_line_snr_threshold=Weak_FeVII_SNR_Threshold,
				main_fevii_flag=Detected_FeVII_Flag
			)

			Candidate_Score = Candidate_Score + LineScore[0]
			Spectral_Lines_For_Analysis[item][9] = LineScore[4]
			Spectral_Lines_For_Analysis[item][10] = LineScore[2]
			Spectral_Lines_For_Analysis[item][11] = LineScore[7]
			Spectral_Lines_For_Analysis[item][20] = LineScore[9]

			Secondary_FeVII_Line_Flag = Secondary_FeVII_Line_Flag + LineScore[10]

			if Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]6087':
				if LineScore[0] != 0:
					Detected_FeVII_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeX]6375':
				if LineScore[0] != 0:
						Detected_FeX_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeXI]7892':
				if LineScore[0] != 0:
					Detected_FeXI_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeXIV]5303':
				if LineScore[0] != 0:
					Detected_FeXIV_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]3759':
				if LineScore[0] != 0:
					Detected_FeVII_3759_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]5159':
				if LineScore[0] != 0:
					Detected_FeVII_5159_Flag = 1

			elif  Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]5721':
				if LineScore[0] != 0:
					Detected_FeVII_5721_Flag = 1

		else:
			Spectral_Lines_For_Analysis[item][9] = np.nan
			Spectral_Lines_For_Analysis[item][10] = np.nan
			Spectral_Lines_For_Analysis[item][11] = np.nan

			if Relative_max_snr >= Strong_SNR_Threshold:
				Spectral_Lines_For_Analysis[item][20] = 1
			else:
				Spectral_Lines_For_Analysis[item][20] = 0

	if not ignore_score_check:

		if Candidate_Score != Recorded_Database_Score:
			print(
				"Candidate Score does NOT match the value recorded in the database.\n"
				f"ID: {ID}\n"
				f"Current Score: {Candidate_Score}\tRecorded Score: {Recorded_Database_Score}"
				"\nThis is likely due to a parameter redefinition between the running of this script and the "
				"last time the analysis script was run."
				"\nIt can also be caused by a change or enabled/disabled spectral smoothing"
				"\nI 'suggest' the analysis script be rerun now."
			)

			if Line_Report:
				line_scoring_report.close()

			sys.exit(1)

	Plotting_Functions.classification_plot(
			object_name=ID,
			object_id=SpectrumID,
			object_tile=ID,
			object_spectrum=[Wave, Flux, FluxErr],
			mean_snr = Mean_SNR,
			object_data=Object_Data,
			line_labels=Spec_Labels,
			line_names=Spec_Names,
			lines_to_plot=Spec_Lines,
			line_data=Spectral_Lines_For_Analysis,
			problematic_skylines = problematic_sky_list_z_corrected,
			skylines=Skylines,
			score=Candidate_Score,
			save_path = Classification_Plots_SavePath,
			z=Redshift,
			date=Observation_Date,
			text_flag=True,
			plot_skylines=True,
			lower_shift = Lower_Shift,
			upper_shift = Upper_Shift,
			candidate_threshold = Candidate_Threshold,
			candidate_score_max = Candidate_Score_Max
		)
